# 01 — Ingestão da camada Bronze

## Objetivo

Este notebook realiza a ingestão das nove fontes brutas para tabelas Delta na camada Bronze.

## Estratégia adotada

- preservar os dados com o mínimo possível de transformação
- manter estruturas aninhadas dos arquivos JSON
- preservar duplicidades existentes nas fontes
- adicionar metadados técnicos de rastreabilidade
- utilizar carga completa com sobrescrita controlada
- validar a quantidade de registros antes e depois da persistência

A carga completa foi escolhida porque o case disponibiliza um único snapshot

In [0]:
# ============================================================
# CONFIGURAÇÕES
# ============================================================

import os
import uuid
from datetime import datetime, timezone

import pandas as pd

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

spark.conf.set(
    "spark.sql.session.timeZone",
    "America/Sao_Paulo"
)

BATCH_ID = str(uuid.uuid4())
BATCH_TIMESTAMP = datetime.now(timezone.utc)

SCHEMA_LANDING = "case_data_engineer"
SCHEMA_BRONZE = "case_data_engineer_bronze"
VOLUME_LANDING = "landing"

print(f"Batch ID: {BATCH_ID}")
print(f"Timestamp UTC: {BATCH_TIMESTAMP}")

In [0]:
# ============================================================
# LOCALIZAÇÃO DO VOLUME
# ============================================================

catalogos_disponiveis = [
    linha["catalog"]
    for linha in spark.sql("SHOW CATALOGS").collect()
]

catalogos_ignorados = {
    "system",
    "samples",
    "hive_metastore"
}

CATALOG = None
erros_catalogos = {}

for catalogo in catalogos_disponiveis:

    if catalogo.lower() in catalogos_ignorados:
        continue

    try:
        spark.sql(
            f"""
            DESCRIBE VOLUME
            `{catalogo}`.`{SCHEMA_LANDING}`.`{VOLUME_LANDING}`
            """
        ).collect()

        CATALOG = catalogo
        break

    except Exception as erro:
        erros_catalogos[catalogo] = str(erro)[:300]

if CATALOG is None:
    raise RuntimeError(
        "O Volume de landing não foi localizado, execute primeiro o notebook 00_exploracao_fontes"
    )

VOLUME_PATH = (
    f"/Volumes/"
    f"{CATALOG}/"
    f"{SCHEMA_LANDING}/"
    f"{VOLUME_LANDING}"
)

print(f"Catálogo localizado: {CATALOG}")
print(f"Volume localizado: {VOLUME_PATH}")

In [0]:
# ============================================================
# VALIDAÇÃO DA LANDING
# ============================================================

ARQUIVOS_ESPERADOS = [
    "atendimento_ocorrencias.ndjson",
    "cadastro_produtos_api_dump.json",
    "comercial_canais.xlsx",
    "crm_clientes_export.xlsx",
    "erp_pedidos_cabecalho_2025.csv",
    "erp_pedidos_itens_2025.csv",
    "legado_regioes_pipe.txt",
    "logistica_entregas.json",
    "vendedores.csv"
]

arquivos_encontrados = {
    arquivo.name
    for arquivo in os.scandir(VOLUME_PATH)
    if arquivo.is_file()
}

arquivos_ausentes = sorted(
    set(ARQUIVOS_ESPERADOS)
    - arquivos_encontrados
)

if arquivos_ausentes:
    raise FileNotFoundError(
        "A landing está incompleta.."
        + "\n- ".join(arquivos_ausentes)
    )

print(
    f"Landing validada: "
    f"{len(ARQUIVOS_ESPERADOS)}/"
    f"{len(ARQUIVOS_ESPERADOS)} arquivos"
)

In [0]:
# ============================================================
# CRIAÇÃO DO SCHEMA BRONZE
# ============================================================

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    `{CATALOG}`.`{SCHEMA_BRONZE}`
    COMMENT 'Camada Bronze do case técnico de Engenharia de Dados'
    """
)

spark.sql(
    f"""
    USE CATALOG `{CATALOG}`
    """
)

print(
    f"Schema Bronze disponível: "
    f"{CATALOG}.{SCHEMA_BRONZE}"
)

In [0]:
# ============================================================
# FUNÇÕES DE LEITURA BRUTA
# ============================================================

def caminho_landing(nome_arquivo: str) -> str:
    return f"{VOLUME_PATH}/{nome_arquivo}"

# Adiciona metadados técnicos sem alterar os campos da origem
def adicionar_metadados(
    df: DataFrame,
    nome_arquivo: str
) -> DataFrame:

    colunas_origem = [
        F.col(f"`{coluna}`")
        for coluna in df.columns
    ]
    return (
        df
        .withColumn(
            "_source_record_hash",
            F.sha2(
                F.to_json(
                    F.struct(*colunas_origem)
                ),
                256
            )
        )
        .withColumn(
            "_source_file",
            F.lit(nome_arquivo)
        )
        .withColumn(
            "_source_path",
            F.lit(caminho_landing(nome_arquivo))
        )
        .withColumn(
            "_ingestion_batch_id",
            F.lit(BATCH_ID)
        )
        .withColumn(
            "_ingestion_timestamp_utc",
            F.lit(BATCH_TIMESTAMP)
        )
    )


def ler_csv_raw(
    nome_arquivo: str,
    separador: str
) -> DataFrame:

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", separador)
        .option("encoding", "UTF-8")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .load(caminho_landing(nome_arquivo))
    )

    return adicionar_metadados(
        df,
        nome_arquivo
    )


def ler_json_raw(
    nome_arquivo: str,
    multiline: bool
) -> DataFrame:

    df = (
        spark.read
        .format("json")
        .option(
            "multiLine",
            str(multiline).lower()
        )
        .option("primitivesAsString", "true")
        .option("dropFieldIfAllNull", "false")
        .option("mode", "PERMISSIVE")
        .load(caminho_landing(nome_arquivo))
    )

    return adicionar_metadados(
        df,
        nome_arquivo
    )


def ler_excel_raw(
    nome_arquivo: str,
    nome_planilha=0
) -> DataFrame:

    caminho = caminho_landing(nome_arquivo)

    pdf = pd.read_excel(
        caminho,
        sheet_name=nome_planilha,
        dtype=object
    )

    pdf.columns = [
        str(coluna).strip()
        for coluna in pdf.columns
    ]

    registros = [
        tuple(
            None if pd.isna(valor) else str(valor)
            for valor in linha
        )
        for linha in pdf.itertuples(
            index=False,
            name=None
        )
    ]

    schema = StructType([
        StructField(
            str(coluna),
            StringType(),
            True
        )
        for coluna in pdf.columns
    ])

    df = spark.createDataFrame(
        registros,
        schema=schema
    )

    return adicionar_metadados(
        df,
        nome_arquivo
    )


print("Funções de leitura disponíveis")

In [0]:
# ============================================================
# CARREGAMENTO DAS FONTES
# ============================================================

fontes_bronze = {
    "bronze_pedidos_cabecalho": ler_csv_raw(
        "erp_pedidos_cabecalho_2025.csv",
        separador=";"
    ),

    "bronze_pedidos_itens": ler_csv_raw(
        "erp_pedidos_itens_2025.csv",
        separador=","
    ),

    "bronze_vendedores": ler_csv_raw(
        "vendedores.csv",
        separador=";"
    ),

    "bronze_regioes": ler_csv_raw(
        "legado_regioes_pipe.txt",
        separador="|"
    ),

    "bronze_produtos": ler_json_raw(
        "cadastro_produtos_api_dump.json",
        multiline=True
    ),

    "bronze_entregas": ler_json_raw(
        "logistica_entregas.json",
        multiline=True
    ),

    "bronze_ocorrencias": ler_json_raw(
        "atendimento_ocorrencias.ndjson",
        multiline=False
    ),

    "bronze_clientes": ler_excel_raw(
        "crm_clientes_export.xlsx",
        nome_planilha=0
    ),

    "bronze_canais": ler_excel_raw(
        "comercial_canais.xlsx",
        nome_planilha="canais"
    )
}

print("Fontes carregadas para a Bronze: ")

for tabela in fontes_bronze:
    print(f"- {tabela}")

In [0]:
# ============================================================
# TABELAS BRONZE
# ============================================================

resultado_carga = []

for nome_tabela, df in fontes_bronze.items():

    tabela_completa = (
        f"`{CATALOG}`."
        f"`{SCHEMA_BRONZE}`."
        f"`{nome_tabela}`"
    )

    quantidade_origem = df.count()

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_completa)
    )

    quantidade_destino = (
        spark.table(tabela_completa)
        .count()
    )

    status = (
        "SUCESSO"
        if quantidade_origem == quantidade_destino
        else "DIVERGENCIA"
    )

    resultado_carga.append(
        (
            nome_tabela,
            quantidade_origem,
            quantidade_destino,
            status,
            BATCH_ID,
            BATCH_TIMESTAMP
        )
    )

    print(
        f"{nome_tabela}: "
        f"{quantidade_origem} -> "
        f"{quantidade_destino} | "
        f"{status}"
    )

In [0]:
# ============================================================
# AUDITORIA DA INGESTÃO
# ============================================================

schema_auditoria = """
    tabela STRING,
    quantidade_origem LONG,
    quantidade_destino LONG,
    status STRING,
    ingestion_batch_id STRING,
    ingestion_timestamp_utc TIMESTAMP
"""

df_auditoria_bronze = spark.createDataFrame(
    resultado_carga,
    schema=schema_auditoria
)

display(
    df_auditoria_bronze
    .orderBy("tabela")
)

In [0]:
# ============================================================
# ERRO DE INGESTÃO
# ============================================================

cargas_com_erro = (
    df_auditoria_bronze
    .filter(
        F.col("status") != "SUCESSO"
    )
    .count()
)

if cargas_com_erro > 0:
    raise RuntimeError(
        f"Foram encontradas {cargas_com_erro} cargas com divergência de quantidade! "
    )

print(
    "Camada Bronze criada com sucesso, todas as contagens OK"
)

In [0]:
# ============================================================
# VALIDAÇÃO DA AUDITORIA
# ============================================================

TABELA_AUDITORIA = (
    f"`{CATALOG}`."
    f"`{SCHEMA_BRONZE}`."
    f"`bronze_ingestion_audit`"
)

(
    df_auditoria_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(TABELA_AUDITORIA)
)

print(
    f"Auditoria registrada em: "
    f"{CATALOG}.{SCHEMA_BRONZE}."
    "bronze_ingestion_audit"
)

In [0]:
# ============================================================
# CHECK TABELAS DISPONÍVEIS NA BRONZE
# ============================================================

display(
    spark.sql(
        f"""
        SHOW TABLES IN
        `{CATALOG}`.`{SCHEMA_BRONZE}`
        """
    )
)

# Conclusão da camada Bronze

As nove fontes foram ingeridas com sucesso e persistidas como tabelas Delta.
A camada Bronze preserva os dados recebidos, incluindo duplicidades, estruturas aninhadas e inconsistências identificadas durante a exploração das fontes.

Foram adicionados metadados técnicos para rastreabilidade:

- arquivo de origem
- caminho de origem
- identificador do lote
- timestamp da ingestão
- hash do registro

A conciliação entre as quantidades de origem e destino foi concluída sem divergências.

A tabela de auditoria mantém o histórico das execuções, enquanto as tabelas Bronze representam o snapshot mais recente das fontes disponibilizadas.